# Tiny LSTM Movie Sentiment Analysis 🎬

## Can an AI read a movie review and decide whether it is positive or negative?

We will build:

**words → Embedding → LSTM → sentiment probability**

We use the **IMDB movie review dataset** included with Keras. Labels: `0 = negative`, `1 = positive`.

At the end, you can type your **own movie review** and ask the trained LSTM to judge it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import re

print("TensorFlow version:", tf.__version__)


# 1. Load the IMDB movie review dataset

We keep only the **10,000 most common words**. Keras has already converted each review from words into integer IDs.

For example, instead of `this movie was amazing`, the neural network sees integer IDs. Later, the **Embedding layer** learns a vector for each word ID.


In [ ]:
VOCAB_SIZE = 10_000
MAX_LENGTH = 200

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print("Training reviews:", len(X_train))
print("Testing reviews: ", len(X_test))
print("First review contains", len(X_train[0]), "tokens")
print("First 20 word IDs:", X_train[0][:20])
print("Label:", y_train[0])


# 2. Turn the numbers back into words

This is only for us humans to inspect the data. The model itself still works with numbers.


In [ ]:
word_index = keras.datasets.imdb.get_word_index()

# Keras reserves 0, 1, 2, 3 for special purposes.
id_to_word = {index + 3: word for word, index in word_index.items()}
id_to_word[0] = "<PAD>"
id_to_word[1] = "<START>"
id_to_word[2] = "<UNKNOWN>"
id_to_word[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(id_to_word.get(int(i), "?") for i in encoded_review)

print(decode_review(X_train[0])[:1000])
print("\nSentiment:", "POSITIVE" if y_train[0] == 1 else "NEGATIVE")


# 3. Make every review the same length

We use the **last 200 tokens** of each review:
- shorter review → add `<PAD>` zeros at the beginning
- longer review → keep its last 200 tokens

So every example has shape `(200,)`.


In [ ]:
X_train = keras.utils.pad_sequences(
    X_train, maxlen=MAX_LENGTH, padding="pre", truncating="pre"
)
X_test = keras.utils.pad_sequences(
    X_test, maxlen=MAX_LENGTH, padding="pre", truncating="pre"
)

print("X_train shape:", X_train.shape)
print("One review shape:", X_train[0].shape)


# 4. Build the model

There are only **three important layers**:

### Embedding
Turns each word ID into a small learned vector.

### LSTM
Reads those vectors **in order** and builds a memory of the review.

### Dense
Turns the final LSTM output into one probability between 0 and 1.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LENGTH,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=32, mask_zero=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])

model.summary()


# 5. Train

This is binary classification, so we use **Binary Crossentropy**. We train for only 3 epochs to keep the classroom run reasonably fast.


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


# 6. Did the model learn?


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], marker="o", label="Training")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curve")
plt.ylim(0.5, 1.0)
plt.legend()
plt.grid(alpha=0.2)
plt.show()

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.1%}")


# 7. Test a real IMDB review

The output is a probability: near **0** means negative; near **1** means positive.


In [ ]:
example_number = 10
probability = float(model.predict(X_test[example_number:example_number+1], verbose=0)[0, 0])

print(decode_review(X_test[example_number])[-1200:])
print()
print(f"Model probability of POSITIVE: {probability:.1%}")
print("Model prediction:", "POSITIVE 👍" if probability >= 0.5 else "NEGATIVE 👎")
print("True label:", "POSITIVE" if y_test[example_number] == 1 else "NEGATIVE")


# 8. How do we let students type their own review?

The model cannot read letters directly. We repeat the same conversion:

**sentence → words → word IDs → padding → Embedding → LSTM**


In [ ]:
# Same vocabulary used by the IMDB dataset.
word_to_id = {word: index + 3 for word, index in word_index.items()}

def encode_text(text):
    # Simple classroom tokenizer: lowercase and keep letters/apostrophes.
    words = re.findall(r"[a-z']+", text.lower())

    ids = [1]  # <START>
    for word in words:
        token_id = word_to_id.get(word, 2)
        # Words outside our 10,000-word vocabulary become <UNKNOWN>.
        ids.append(token_id if token_id < VOCAB_SIZE else 2)

    return keras.utils.pad_sequences(
        [ids], maxlen=MAX_LENGTH, padding="pre", truncating="pre"
    )

print("Input shape:", encode_text("This movie was amazing!").shape)


# 9. Try your own review 🎮

Type a review and click **Analyze sentiment**.

Try:
- `This movie was amazing.`
- `This movie was not amazing.`
- `The movie was terrible, but the ending was amazing.`
- `I expected an amazing movie, but it was boring and disappointing.`

Watch how probability changes as **sequence and context** change.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

review_box = widgets.Textarea(
    value="The movie started slowly, but I absolutely loved the ending.",
    placeholder="Type your movie review here...",
    description="Review:",
    layout=widgets.Layout(width="90%", height="100px")
)
analyze_button = widgets.Button(description="Analyze sentiment", button_style="info")
result_output = widgets.Output()

def analyze_review(button):
    encoded = encode_text(review_box.value)
    probability = float(model.predict(encoded, verbose=0)[0, 0])

    with result_output:
        clear_output(wait=True)
        print(f"POSITIVE probability: {probability:.1%}")
        print(f"NEGATIVE probability: {1-probability:.1%}")
        print()
        print("👍 Model prediction: POSITIVE" if probability >= 0.5
              else "👎 Model prediction: NEGATIVE")

analyze_button.on_click(analyze_review)

display(widgets.HTML("<h3>🎬 LSTM Movie Review Analyzer</h3>"))
display(review_box)
display(analyze_button)
display(result_output)
analyze_review(None)


# 10. A sequence experiment: watch the opinion change

We start negative and gradually add a positive ending. The **same trained model** evaluates every partial sentence.


In [ ]:
sentences = [
    "The movie",
    "The movie was",
    "The movie was terrible",
    "The movie was terrible but",
    "The movie was terrible but the ending",
    "The movie was terrible but the ending was amazing",
    "The movie was terrible but the ending was absolutely amazing",
]

for sentence in sentences:
    p = float(model.predict(encode_text(sentence), verbose=0)[0, 0])
    print(f"{p:6.1%} positive  |  {sentence}")


# 11. What did we learn?

A movie review is not just a bag of positive and negative words.

Compare:
- **amazing**
- **not amazing**
- **terrible, but the ending was amazing**

The **order and context** matter.

Our pipeline was:

`words → word IDs → Embedding → LSTM → sentiment`

The Embedding layer learns numerical representations of words. The LSTM reads those representations as a **sequence** and carries information forward through the review.

## Next question

**What exactly is an embedding, and why should similar words have similar vectors?**

That is our bridge to the next lesson.
